# 🚀 Liftoff Ultimate — Render Rápido Gratis en Google Colab (GPU)

El notebook definitivo para renderizar proyectos de **Blender** gratis usando aceleración por GPU en **Google Colab**.

### ✨ Características Principales:
- ⚡ **Auto-detección inteligente:** Localiza automáticamente tu archivo `*_cloud.blend` más reciente en tu Google Drive.
- 🎨 **Multi-versión de Blender:** Compatible con **Blender 4.2 LTS**, 4.1.1, 4.0.2, 3.6 LTS, 3.3 LTS y 2.93 LTS.
- ⚡ **Optimizador de GPU en tiempo real:** Autodetecta OptiX / CUDA para máximo rendimiento en NVIDIA T4/V100/A100.
- 🎛️ **Overrides sin reabrir Blender:** Modifica muestras (samples), porcentaje de resolución o motor de render (Cycles / EEVEE) sobre la marcha.
- 🎬 **Modo Frame o Animación:** Renderiza imágenes fijas o secuencias completas de animación.
- 💡 **Soporte Headless EEVEE (`xvfb`):** Evita cierres inesperados al renderizar escenas con EEVEE Next sin pantalla gráfica.
- 🖼️ **Vista previa al instante:** Muestra el resultado final directamente en el cuaderno.
- 🔔 **Notificaciones opcionales:** Envía un aviso a Discord cuando tu render termine.

---

In [ ]:
# 1) Verificación de la GPU asignada por Google Colab
!nvidia-smi -L

In [ ]:
# 2) Montar Google Drive (donde Liftoff sincroniza tu .blend)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 🎛️ 3) Configuración Global del Render { display-mode: "form" }
import os, glob

# --- AUTO-DETECCIÓN DE ARCHIVO .BLEND ---
drive_renders_dir = "/content/drive/MyDrive/RENDERS"
found_blend_files = glob.glob(os.path.join(drive_renders_dir, "*_cloud.blend")) + glob.glob(os.path.join(drive_renders_dir, "*.blend"))

if found_blend_files:
    found_blend_files.sort(key=os.path.getmtime, reverse=True)
    auto_blend = found_blend_files[0]
    print(f"✨ Archivo .blend autodetectado más reciente: {auto_blend}")
else:
    auto_blend = "/content/drive/MyDrive/RENDERS/mi_escena_cloud.blend"
    print("⚠️ No se encontraron archivos .blend en Drive/RENDERS. Ingresa la ruta manualmente.")

#@markdown --- 
#@markdown ### 📂 Rutas de Archivos
BLEND_FILE_INPUT = "AUTO" #@param {type:"string"}
BLEND_FILE = auto_blend if BLEND_FILE_INPUT.strip().upper() == "AUTO" else BLEND_FILE_INPUT
OUTPUT_DIR = "/content/drive/MyDrive/RENDERS/out/" #@param {type:"string"}

#@markdown --- 
#@markdown ### 🎬 Modo de Renderizado
RENDER_MODE = "Frame Est\u00e1tico" #@param ["Frame Est\u00e1tico", "Animaci\u00f3n (Rango de Frames)"]
FRAME = 1 #@param {type:"integer"}
START_FRAME = 1 #@param {type:"integer"}
END_FRAME = 250 #@param {type:"integer"}

#@markdown --- 
#@markdown ### ⚡ Ajustes de Rendimiento y Overrides (Opcionales)
RENDER_ENGINE = "Mantener Escena" #@param ["Mantener Escena", "CYCLES", "BLENDER_EEVEE_NEXT"]
OVERRIDE_SAMPLES = 0 #@param {type:"integer"}  # 0 = Usar muestras guardadas en la escena (ej: 64, 128, 256, 512)
RESOLUTION_SCALE = 100 #@param {type:"integer"} # 100 = 100%, 50 = 50% de la resolución original
ENABLE_DENOISER = True #@param {type:"boolean"}

#@markdown --- 
#@markdown ### 📦 Versión de Blender
BLENDER_VERSION = "4.2.0 (LTS Recomendada)" #@param ["4.2.0 (LTS Recomendada)", "4.1.1", "4.0.2", "3.6.13 (LTS)", "3.3.20 (LTS)", "2.93.18 (LTS)"]

#@markdown --- 
#@markdown ### 🔔 Notificaciones (Opcional)
DISCORD_WEBHOOK_URL = "" #@param {type:"string"}

version_urls = {
    "4.2.0 (LTS Recomendada)": "https://download.blender.org/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz",
    "4.1.1": "https://download.blender.org/release/Blender4.1/blender-4.1.1-linux-x64.tar.xz",
    "4.0.2": "https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz",
    "3.6.13 (LTS)": "https://download.blender.org/release/Blender3.6/blender-3.6.13-linux-x64.tar.xz",
    "3.3.20 (LTS)": "https://download.blender.org/release/Blender3.3/blender-3.3.20-linux-x64.tar.xz",
    "2.93.18 (LTS)": "https://download.blender.org/release/Blender2.93/blender-2.93.18-linux-x64.tar.xz"
}
BLENDER_URL = version_urls.get(BLENDER_VERSION, version_urls["4.2.0 (LTS Recomendada)"])

print(f"\n🎯 CONFIGURACIÓN CARGADA:")
print(f"- Archivo: {BLEND_FILE}")
print(f"- Salida: {OUTPUT_DIR}")
print(f"- Modo: {RENDER_MODE}")
print(f"- Versión Blender: {BLENDER_VERSION}")

In [ ]:
# 4) Instalación de Blender + Librerías Headless (xvfb)
import os
os.makedirs('/content/blender', exist_ok=True)

print("📥 Instalando dependencias del sistema y pantalla virtual (xvfb)...")
!apt-get update -qq && apt-get install -y -qq xvfb libxxf86vm1 libgl1-mesa-glx libxi6 libxrender1 > /dev/null 2>&1

print(f"📦 Descargando Blender desde {BLENDER_URL}...")
!wget -q --show-progress -O /content/blender.tar.xz "$BLENDER_URL"
!tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1

print("✅ Blender listo:")
!/content/blender/blender --version

In [ ]:
# 5) Generador de Script de Optimización GPU & Overrides
script_content = f"""
import bpy

# 1. Configuración de Cómputo GPU (OptiX / CUDA)
try:
    prefs = bpy.context.preferences.addons['cycles'].preferences
    elegido = None
    for dt in ('OPTIX', 'CUDA'):
        try:
            prefs.compute_device_type = dt
            elegido = dt
            break
        except Exception:
            continue
    prefs.get_devices()
    for d in prefs.devices:
        d.use = (d.type != 'CPU')
    print(f'🚀 Cómputo Cycles configurado en: {{elegido}} | Dispositivos: {{[d.name for d in prefs.devices if d.use]}}')
except Exception as e:
    print(f'⚠️ No se pudo configurar Cycles GPU Preferences: {{e}}')

# 2. Aplicar ajustes en la escena
for scene in bpy.data.scenes:
    scene.cycles.device = 'GPU'
    
    # Overrides opcionales
    if '{RENDER_ENGINE}' != 'Mantener Escena':
        scene.render.engine = '{RENDER_ENGINE}'
        print(f'⚙️ Motor cambiado a: {{scene.render.engine}}')
        
    if {OVERRIDE_SAMPLES} > 0:
        if hasattr(scene.cycles, 'samples'):
            scene.cycles.samples = {OVERRIDE_SAMPLES}
            print(f'⚙️ Muestras (Samples) fijadas en: {{scene.cycles.samples}}')
            
    if {RESOLUTION_SCALE} != 100:
        scene.render.resolution_percentage = {RESOLUTION_SCALE}
        print(f'⚙️ Escala de resolución ajustada al: {{scene.render.resolution_percentage}}%')
        
    if {ENABLE_DENOISER}:
        if hasattr(scene.cycles, 'use_denoising'):
            scene.cycles.use_denoising = True
            print('⚙️ Denoiser de Cycles activado')
"""

with open('/content/optimize_render.py', 'w', encoding='utf-8') as f:
    f.write(script_content)

print("⚡ Script /content/optimize_render.py generado correctamente.")

In [ ]:
# 6) Ejecución del Renderizado
import os, time
os.makedirs(OUTPUT_DIR, exist_ok=True)

start_time = time.time()

if RENDER_MODE == "Frame Estático":
    print(f"🎬 Renderizando Frame {FRAME}...")
    !xvfb-run -a /content/blender/blender -b "$BLEND_FILE" -P /content/optimize_render.py -o "$OUTPUT_DIR" -f $FRAME
else:
    print(f"🎞️ Renderizando Animación (Frames {START_FRAME} a {END_FRAME})...")
    !xvfb-run -a /content/blender/blender -b "$BLEND_FILE" -P /content/optimize_render.py -o "$OUTPUT_DIR" -s $START_FRAME -e $END_FRAME -a

elapsed = time.time() - start_time
print(f"\n⏱️ Renderizado completado en {elapsed:.2f} segundos.")

In [ ]:
# 7) Vista Previa & Notificación Opcional a Discord
import os, glob, requests
from IPython.display import display, Image

# Galería de vista previa del último renderizado
rendered_files = glob.glob(os.path.join(OUTPUT_DIR, "*.png")) + glob.glob(os.path.join(OUTPUT_DIR, "*.jpg"))
if rendered_files:
    rendered_files.sort(key=os.path.getmtime, reverse=True)
    latest_render = rendered_files[0]
    print(f"🖼️ Vista previa del último render: {latest_render}")
    display(Image(filename=latest_render, width=650))
else:
    print("ℹ️ No se encontraron archivos de imagen en la carpeta de salida.")

# Envío de notificación por Discord Webhook (si se ingresó URL)
if DISCORD_WEBHOOK_URL.strip():
    try:
        payload = {"content": f"🚀 **Liftoff Render Completado!**\n📂 Archivo: `{os.path.basename(BLEND_FILE)}`\n📁 Carpeta: `{OUTPUT_DIR}`"}
        response = requests.post(DISCORD_WEBHOOK_URL, json=payload)
        if response.status_code in (200, 204):
            print("🔔 Notificación enviada exitosamente a Discord.")
        else:
            print(f"⚠️ Error al enviar webhook a Discord (Código {response.status_code})")
    except Exception as e:
        print(f"⚠️ No se pudo enviar la notificación: {e}")

## ¡Render Completado con Éxito! ✅

Tus archivos se guardaron directamente en la carpeta `OUTPUT_DIR` de tu Google Drive y ya están sincronizándose automáticamente con tu equipo.

**Créditos:** Creado por **yomero243** ([blender-liftoff](https://github.com/yomero243/blender-liftoff)) combinando lo mejor del renderizado rápido gratis en la nube.